# 02 — Feature Engineering & Selection
Create derived features, then use Information Value to select the final 15–20 for modelling.

In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

PROC = Path('../Data/Processed')

master = pd.read_parquet(PROC / 'master.parquet')

## 1. Derived features

In [2]:
# affordability ratios
master['credit_annuity_ratio'] = master['AMT_CREDIT'] / master['AMT_ANNUITY']
master['income_annuity_ratio'] = master['AMT_INCOME_TOTAL'] / master['AMT_ANNUITY']
master['income_credit_ratio']  = master['AMT_INCOME_TOTAL'] / master['AMT_CREDIT']
master['credit_goods_ratio']   = master['AMT_CREDIT'] / master['AMT_GOODS_PRICE']

# age and employment — sentinel 365243 = unemployed, not a real value
master['age_years']          = master['DAYS_BIRTH'] / -365
master['is_unemployed']      = (master['DAYS_EMPLOYED'] == 365243).astype(int)
master['employed_years']     = master['DAYS_EMPLOYED'].where(master['DAYS_EMPLOYED'] != 365243) / -365
master['employed_age_ratio'] = master['employed_years'] / master['age_years']

# external score composites across all three sources
master['ext_source_mean'] = master[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)
master['ext_source_min']  = master[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].min(axis=1)

# bureau overdue as a share of total credit
master['bureau_overdue_rate'] = master['bureau_total_overdue'] / master['bureau_total_credit'].replace(0, pd.NA)

# address instability and recent credit inquiries
master['address_mismatch'] = master['REG_CITY_NOT_LIVE_CITY'] + master['REG_CITY_NOT_WORK_CITY']
master['recent_inquiries'] = master['AMT_REQ_CREDIT_BUREAU_MON'] + master['AMT_REQ_CREDIT_BUREAU_QRT']

print(f"Master shape after feature engineering: {master.shape}")

Master shape after feature engineering: (307511, 74)


## 2. Information Value — feature selection

In [3]:
def compute_iv(df, target, n_bins=10):
    rows = []
    for col in df.columns:
        if col == target:
            continue
        tmp = df[[col, target]].copy()
        # bin numeric columns into quantiles; leave categoricals as-is
        if tmp[col].dtype.kind in 'iuf':
            tmp[col] = pd.qcut(tmp[col], q=n_bins, duplicates='drop')
        grouped = tmp.groupby(col, observed=True)[target].agg(['sum', 'count'])
        grouped.columns = ['events', 'total']
        grouped['non_events'] = grouped['total'] - grouped['events']
        # drop empty bins to avoid log(0)
        grouped = grouped[(grouped['events'] > 0) & (grouped['non_events'] > 0)]
        total_e  = grouped['events'].sum()
        total_ne = grouped['non_events'].sum()
        grouped['pct_e']  = grouped['events']     / total_e
        grouped['pct_ne'] = grouped['non_events'] / total_ne
        iv = ((grouped['pct_e'] - grouped['pct_ne']) * np.log(grouped['pct_e'] / grouped['pct_ne'])).sum()
        rows.append({'variable': col, 'info_value': round(iv, 6)})
    return pd.DataFrame(rows)

iv_table = compute_iv(master.drop(columns='SK_ID_CURR'), target='TARGET')
iv_table = iv_table.sort_values('info_value', ascending=False).reset_index(drop=True)
print(iv_table.to_string())

                      variable  info_value
0              ext_source_mean    0.608650
1               ext_source_min    0.466273
2                 EXT_SOURCE_3    0.410124
3                 EXT_SOURCE_1    0.346526
4                 EXT_SOURCE_2    0.307124
5               cc_utilisation    0.278256
6               cc_avg_balance    0.147522
7         credit_annuity_ratio    0.143627
8              bureau_avg_days    0.132228
9                inst_max_late    0.110965
10           bureau_debt_ratio    0.108262
11               DAYS_EMPLOYED    0.101263
12              cc_avg_payment    0.099306
13              employed_years    0.092529
14             AMT_GOODS_PRICE    0.092017
15             OCCUPATION_TYPE    0.086243
16                  DAYS_BIRTH    0.084200
17                   age_years    0.084176
18              inst_late_rate    0.076485
19          credit_goods_ratio    0.075766
20               inst_avg_diff    0.075098
21           ORGANIZATION_TYPE    0.073368
22         

## 3. Final selected features

In [4]:
# raw columns superseded by derived ones — exclude to avoid duplication
exclude = {
    'SK_ID_CURR', 'TARGET',
    'DAYS_BIRTH',                                        # → age_years
    'DAYS_EMPLOYED',                                     # → employed_years, is_unemployed
    'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', # → address_mismatch
    'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT', # → recent_inquiries
    'bureau_total_overdue', 'bureau_total_credit',       # → bureau_overdue_rate
}

# keep features with IV > 0.05 that are not in the exclude list
selected_features = (
    iv_table
    .loc[iv_table['info_value'] > 0.05, 'variable']
    .loc[~iv_table['variable'].isin(exclude)]
    .tolist()
)

print(f"Selected {len(selected_features)} features:")
for f in selected_features:
    iv = iv_table.loc[iv_table['variable'] == f, 'info_value'].values[0]
    print(f"  {f:35s}  IV = {iv:.4f}")

Selected 25 features:
  ext_source_mean                      IV = 0.6087
  ext_source_min                       IV = 0.4663
  EXT_SOURCE_3                         IV = 0.4101
  EXT_SOURCE_1                         IV = 0.3465
  EXT_SOURCE_2                         IV = 0.3071
  cc_utilisation                       IV = 0.2783
  cc_avg_balance                       IV = 0.1475
  credit_annuity_ratio                 IV = 0.1436
  bureau_avg_days                      IV = 0.1322
  inst_max_late                        IV = 0.1110
  bureau_debt_ratio                    IV = 0.1083
  cc_avg_payment                       IV = 0.0993
  employed_years                       IV = 0.0925
  AMT_GOODS_PRICE                      IV = 0.0920
  OCCUPATION_TYPE                      IV = 0.0862
  age_years                            IV = 0.0842
  inst_late_rate                       IV = 0.0765
  credit_goods_ratio                   IV = 0.0758
  inst_avg_diff                        IV = 0.0751
  ORGANIZ

In [5]:
import json

# save selected features so other notebooks can load them without copy-pasting
with open(PROC / 'selected_features.json', 'w') as f:
    json.dump(selected_features, f)

master.to_parquet(PROC / 'master_features.parquet', index=False)
print(f"Saved master_features.parquet and selected_features.json")

Saved master_features.parquet and selected_features.json
